# Academic Journals - SPECTER2 Embedding-Analyse

Wandelt jedes Paper mit Titel und Abstract in einen Zahlen-Vektor um und misst, wie ähnlich (thematisch) sich Paper sind. Ergebnis: **topic_match**. Abschnitte der Reihe nach ausführen.

## 1 - Was macht SPECTER2?

SPECTER2 verwandelt Titel + Abstract eines Papers in einen Zahlen-Vektor. Ähnliche Themen ergeben ähnliche Vektoren. Damit können wir thematische Nähe zwischen Papern messen.

## 2 - Welche Autoren?

Wir rechnen über alle Autoren mit mindestens 3 Papern. Autoren mit nur einem Paper bringen für die Journal-Treue nichts.

## 3 - Datenbank in Colab

Die DuckDB liegt in Google Drive. Wir binden Drive ein und kopieren die Datei in den lokalen Colab-Speicher (geht schneller). Immer mit GPU laufen lassen!

In [ ]:
# Google Drive verbinden
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Datenbank aus Drive lokal kopieren
import shutil, os

DRIVE_DB = "/content/drive/MyDrive/Academic-Journals/academic_journals.duckdb"
LOCAL_DB = "/content/academic_journals.duckdb"

assert os.path.exists(DRIVE_DB), f"DB nicht gefunden: {DRIVE_DB}  (Pfad anpassen!)"
shutil.copy(DRIVE_DB, LOCAL_DB)
print("DB kopiert nach", LOCAL_DB, f"({os.path.getsize(LOCAL_DB)/1e6:.0f} MB)")

DB kopiert nach /content/academic_journals.duckdb (233 MB)


## 4 - Setup

Pakete installieren und pruefen, ob eine GPU aktiv ist.

In [ ]:
# Pakete installieren + GPU pruefen
!pip -q install duckdb transformers adapters

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Device: cuda


## 5 - Autoren und Paper laden

Alle produktiven Autoren auswählen und ihre Paper aus der Datenbank ziehen.

In [ ]:
# Produktive Autoren finden
import duckdb, numpy as np

TBL         = "openalex_ai_raw_v1_0"   # Rohdatenbasis
MIN_PAPERS  = 3                         # Autor muss mindestens so viele Paper veröffentlicht haben
N_AUTHORS   = 150                       # Zufallsauswahl der Autoren
SEED        = 42

con = duckdb.connect(LOCAL_DB, read_only=True)

# Autor -> Paper auswählen --> nur Paper mit Titel + Abstract
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW author_paper AS
    WITH exploded AS (
        SELECT id AS work_id,
               unnest(authorships) AS a
        FROM {TBL}
        WHERE title IS NOT NULL AND abstract_inverted_index IS NOT NULL
    )
    SELECT a.author.id AS author_id, a.author.display_name AS author_name, work_id
    FROM exploded
    WHERE a.author.id IS NOT NULL
""")

# Nur Autoren auswählen, die auch was veröffentlicht haben
prod = con.execute(f"""
    SELECT author_id, author_name, COUNT(DISTINCT work_id) AS n_paper
    FROM author_paper
    GROUP BY author_id, author_name
    HAVING COUNT(DISTINCT work_id) >= {MIN_PAPERS}
    ORDER BY n_paper DESC
""").fetchall()

print(f"{len(prod)} Autoren mit >= {MIN_PAPERS} Papern.")

rng = np.random.default_rng(SEED)

# Ab jetzt --> Voller Lauf mit allen Autoren
chosen_ids = [p[0] for p in prod]
print(f"{len(chosen_ids)} Autoren (VOLLER LAUF, alle mit >= {MIN_PAPERS} Papern).")

5500 Autoren mit >= 3 Papern.
5500 Autoren (VOLLER LAUF, alle mit >= 3 Papern).


In [ ]:
# Paper der Autoren laden (mit IDs und Datum)
ph = ",".join(["?"] * len(chosen_ids))
rows = con.execute(f"""
    WITH work_ids AS (
        SELECT DISTINCT work_id FROM author_paper WHERE author_id IN ({ph})
    )
    SELECT
        w.id                                          AS work_id,
        w.title,
        w.abstract_inverted_index                     AS abs_idx,
        w.publication_year                            AS year,
        COALESCE(strftime(w.publication_date, '%Y-%m-%d'),
                 CAST(w.publication_year AS VARCHAR) || '-01-01') AS pub_date,
        w.primary_location.source.id                  AS journal_id,
        w.primary_location.source.display_name        AS journal_name,
        w.primary_location.source.host_organization   AS publisher_id,
        w.primary_location.source.is_in_doaj          AS in_doaj
    FROM {TBL} w
    JOIN work_ids ON work_ids.work_id = w.id
""", chosen_ids).fetchall()

print(f"{len(rows)} Paper der gewählten Autoren.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

12237 Paper der gewählten Autoren.


## 6 - Abstract wiederherstellen

OpenAlex speichert Abstracts als Wort-Positionen. Wir setzen daraus wieder normalen Text zusammen.

In [ ]:
# Abstract aus Wort-"Fetzen" zusammensetzen
def reconstruct_abstract(inv):
    if not inv:
        return ""
    pairs = []
    for word, positions in inv.items():
        for p in positions:
            pairs.append((p, word))
    pairs.sort(key=lambda x: x[0])
    return " ".join(w for _, w in pairs)

titles, abstracts, meta = [], [], []
for work_id, title, abs_idx, year, pub_date, journal_id, journal_name, publisher_id, in_doaj in rows:
    abstract = reconstruct_abstract(abs_idx)
    if not abstract.strip():
        continue
    titles.append(title)
    abstracts.append(abstract)
    meta.append({"work_id": work_id, "year": year, "date": pub_date,
                 "journal_id": journal_id, "journal_name": journal_name,
                 "publisher_id": publisher_id, "in_doaj": in_doaj})

print(f"{len(titles)} Paper mit nutzbarem Abstract.")
print("\nBeispiel-Titel   :", titles[0][:90])
print("Beispiel-Abstract:", abstracts[0][:200], "...")

12236 Paper mit nutzbarem Abstract.

Beispiel-Titel   : A survey of transfer learning
Beispiel-Abstract: Machine learning and data mining techniques have been used in numerous real-world applications. An assumption of traditional machine learning methodologies is the training data and testing data are ta ...


## 7 - Paper einbetten

SPECTER2 lädt und macht aus jedem Paper einen Zahlen-Vektor

In [ ]:
# SPECTER2 laden und alle Paper verarbeiten (Embedding)
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

tok   = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoAdapterModel.from_pretrained("allenai/specter2_base")
model.load_adapter("allenai/specter2", source="hf", set_active=True)
model.to(DEVICE).eval()

def embed(titles, abstracts, batch_size=32):
    texts = [(t or "") + tok.sep_token + (a or "") for t, a in zip(titles, abstracts)]
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tok(batch, padding=True, truncation=True,
                  return_tensors="pt", max_length=512).to(DEVICE)
        with torch.no_grad():
            o = model(**inp)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())  # CLS
        print(f"  {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    return np.vstack(out)

X = embed(titles, abstracts)
X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)   # L2-normalisieren
print(f"\nEmbeddings: {X.shape}   (Paper x 768)")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

pytorch_adapter.bin:   0%|          | 0.00/3.59M [00:00<?, ?B/s]

  12236/12236
Embeddings: (12236, 768)   (Paper x 768)


## 8 - Ähnlichkeit Kernel - Matrix

Der RBF-Kernel misst die Ähnlichkeit zwischen zwei Vektoren. Wir rechnen nur die nötigen Vergleiche, nie die riesige Gesamtmatrix - das spart Speicher. Müssen wir aber noch ausführen, warum wir so vorgehen

In [ ]:
# Kernel-Ähnlichkeit, speicherschonend
_rng = np.random.default_rng(0)
_m = min(3000, len(X))
_idx = _rng.choice(len(X), _m, replace=False)
_Xs = X[_idx]
_s = np.sum(_Xs**2, axis=1)
_D2 = np.maximum(_s[:, None] + _s[None, :] - 2 * _Xs @ _Xs.T, 0.0)
sig2 = float(np.median(_D2[np.triu_indices(_m, k=1)]))
print(f"sigma^2 (Median-Heuristik, Stichprobe {_m}) = {sig2:.4f}")

def kernel_group_mean(idxs):
    """Mittlere paarweise RBF-Ähnlichkeit innerhalb einer kleinen Gruppe."""
    if len(idxs) < 2:
        return None
    sub = X[idxs]
    s = np.sum(sub**2, axis=1)
    D2 = np.maximum(s[:, None] + s[None, :] - 2 * sub @ sub.T, 0.0)
    K = np.exp(-0.5 * D2 / sig2)
    iu = np.triu_indices(len(idxs), k=1)
    return float(np.mean(K[iu]))

def sim_to_all(i):
    """RBF-Ähnlichkeit eines Ankers zu ALLEN Papern (ein Vektor, keine Matrix)."""
    d = X - X[i]
    D2 = np.sum(d**2, axis=1)
    return np.exp(-0.5 * D2 / sig2)

# Zentrale Zuordnungen (von hier an notwendig)
row_of        = {m["work_id"]: i for i, m in enumerate(meta)}
journal_id_of = {m["work_id"]: m["journal_id"] for m in meta}
emb_ids       = set(row_of)

sigma^2 (Median-Heuristik, Stichprobe 3000) = 0.3613


## 9 - Testblock

Prüfen, ob ähnliche Paper wirklich thematisch passen, und die "mittlere Ähnlichkeit" von Papern desselben Autors im selben Journal messen.

In [ ]:
# Ähnlichste Paper als Anker ermitteln --> Test
def most_similar(i, k=3):
    sims = sim_to_all(i); sims[i] = -1
    for j in np.argsort(sims)[::-1][:k]:
        print(f"   {sims[j]:.3f}  {titles[j][:75]}")

for a in [0, 1, 2]:
    print(f"\nAnker: {titles[a][:75]}")
    most_similar(a)


Anker: A survey of transfer learning
   0.923  Transfer learning: a friendly introduction
   0.895  A survey on heterogeneous transfer learning
   0.888  Feature-Based Transfer Learning Based on Distribution Similarity

Anker: State-of-the-art in artificial neural network applications: A survey
   0.906  Comprehensive Review of Artificial Neural Network Applications to Pattern R
   0.857  Progress on Artificial Neural Networks for Big Data Analytics: A Survey
   0.853  Local Sigmoid Method: Non-Iterative Deterministic Learning Algorithm for Au

Anker: Survey on deep learning with class imbalance
   0.908  Effective Class-Imbalance Learning Based on SMOTE and Convolutional Neural 
   0.902  Adjusting Decision Boundary for Class Imbalanced Learning
   0.892  A survey on addressing high-class imbalance in big data


In [ ]:
# Q1: mittlere Themen-Ähnlichkeit --> selbes Journal
from collections import defaultdict

author_works = defaultdict(list)
for aid in chosen_ids:
    for (w,) in con.execute(
        "SELECT DISTINCT work_id FROM author_paper WHERE author_id = ?", [aid]
    ).fetchall():
        if w in emb_ids:
            author_works[aid].append(w)

def author_intra_journal(work_ids):
    by_j = defaultdict(list)
    for w in work_ids:
        by_j[journal_id_of[w]].append(row_of[w])
    vals = [kernel_group_mean(idxs) for idxs in by_j.values() if len(idxs) >= 2]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

scores = [(a, author_intra_journal(ws)) for a, ws in author_works.items()]
scores = [(a, s) for a, s in scores if s is not None]
if scores:
    arr = np.array([s for _, s in scores])
    print(f"Autoren mit >=2 Papern im selben Journal: {len(scores)}")
    print(f"Mittlere Intra-Journal-Aehnlichkeit: {arr.mean():.3f} (SD {arr.std():.3f})")
else:
    print("Keine Autoren mit >=2 Papern im selben Journal.")

Autoren mit >=2 Papern im selben Journal: 4366
Mittlere Intra-Journal-Aehnlichkeit: 0.756 (SD 0.084)


## 10 - Was bedeuten die Zahlen?

topic_match liegt zwischen 0 und 1: hoch = thematisch ähnlich, niedrig = fremd. Weil der Datensatz nur KI-Paper enthält, sind die Werte generell hoch - wir lesen **Unterschiede**, nicht absolute Ausprägungen.

## 11 - Anschluss an die anderen Layer

Alle drei Layer verbinden sich über die **work_id**. Unser topic_match ist die Themen-Variable, die der Probabilistic-Layer für die Q3-Frage nutzt.

## 12 - Schlüssel-Tabelle

Alle Autor-Paper-Zeilen mit IDs - die Basis, die der Logic-Layer braucht.

In [ ]:
# Schlüssel-Tabelle (Autor und Paper Zeiln)
keys = con.execute(f"""
    WITH exploded AS (
        SELECT id AS work_id,
               publication_year                            AS year,
               primary_location.source.id                  AS journal_id,
               primary_location.source.display_name        AS journal_name,
               primary_location.source.host_organization   AS publisher_id,
               primary_location.source.is_in_doaj          AS in_doaj,
               unnest(authorships)                          AS a
        FROM {TBL}
    )
    SELECT work_id, year, journal_id, journal_name, publisher_id, in_doaj,
           a.author.id           AS author_id,
           a.author.display_name AS author_name
    FROM exploded
    WHERE a.author.id IS NOT NULL
""").fetchall()

print(f"{len(keys)} Autor-Paper-Zeilen.")
print("Spalten: work_id, year, journal_id, journal_name, publisher_id, in_doaj, author_id, author_name")
print("Beispiel:", keys[0])

110772 Autor-Paper-Zeilen.
Spalten: work_id, year, journal_id, journal_name, publisher_id, in_doaj, author_id, author_name
Beispiel: ('https://openalex.org/W2493916176', 2017, 'https://openalex.org/S2729999759', 'Transactions of the Association for Computational Linguistics', 'https://openalex.org/P4310320244', True, 'https://openalex.org/A5035420035', 'Piotr Bojanowski')


## 13 - Ergebnisse exportieren

Ergebnisse als CSVs ins Google-Drive

In [ ]:
# Export: Q1-Ergebnis als CSV ausgeben
import csv, shutil

rows_out = []
for aid, ws in author_works.items():
    by_j = defaultdict(list)
    for w in ws:
        by_j[journal_id_of[w]].append(w)
    for jid, wids in by_j.items():
        if len(wids) >= 2:
            val = kernel_group_mean([row_of[w] for w in wids])
            rows_out.append({
                "author_id": aid,
                "journal_id": jid,
                "n_papers": len(wids),
                "topic_match_intra": round(val, 4),
                "work_ids": ";".join(wids),
            })

LOCAL_CSV = "/content/results_q1_topic_match.csv"
with open(LOCAL_CSV, "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=["author_id","journal_id","n_papers",
                                       "topic_match_intra","work_ids"])
    wr.writeheader(); wr.writerows(rows_out)

shutil.copy(LOCAL_CSV, "/content/drive/MyDrive/Academic-Journals/results_q1_topic_match.csv")
print(f"{len(rows_out)} Autor-Journal-Zeilen exportiert (voller Lauf).")
if rows_out:
    print("Beispiel:", rows_out[0])

5307 Autor-Journal-Zeilen exportiert (voller Lauf).
Beispiel: {'author_id': 'https://openalex.org/A5048223855', 'journal_id': 'https://openalex.org/S190629608', 'n_papers': 11, 'topic_match_intra': 0.7786, 'work_ids': 'https://openalex.org/W2725188468;https://openalex.org/W2751766815;https://openalex.org/W3156142979;https://openalex.org/W2908959496;https://openalex.org/W2739200628;https://openalex.org/W2411453843;https://openalex.org/W2905816463;https://openalex.org/W2529999104;https://openalex.org/W2582017947;https://openalex.org/W2940772039;https://openalex.org/W4378906427'}


In [ ]:
# Export: Das ist die Schlüssel-Tabelle, ebenfalls als CSV
import csv, shutil

LOCAL_KEYS = "/content/keys_author_paper.csv"
with open(LOCAL_KEYS, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["work_id","year","journal_id","journal_name",
                 "publisher_id","in_doaj","author_id","author_name"])
    wr.writerows(keys)

shutil.copy(LOCAL_KEYS, "/content/drive/MyDrive/Academic-Journals/keys_author_paper.csv")
print(f"{len(keys)} Zeilen exportiert nach keys_author_paper.csv")

110772 Zeilen exportiert nach keys_author_paper.csv


## 14 - topic_match für die Event-Tabelle füllen

Statt eigene Gelegenheiten zu bauen, lesen wir die **Event-Tabelle des Logic-Layers**
ein und füllen nur die leere Spalte topic_match. Pro Zeile (Autor, Journal, Jahr):
Ähnlichkeit zwischen **Autor-Profil** und **Journal-Profil**, beide nur aus Papern
**vor** dem Jahr. Auf **Jahresebene**, wie der Logic-Layer rechnet.

Zeilen ohne Vergangenheits-Profil - oder deren Autor nicht eingebettet ist (nur
produktive Autoren haben Embeddings) - bleiben **leer** (nicht 0). Wichtig für die Weiterverarbeitung!!!

In [ ]:
# Event-Tabelle einlesen (vom Logic-Layer, ueber Drive)  -- dtype-sicher
import pandas as pd
import numpy as np

EVENT_IN  = "/content/drive/MyDrive/Academic-Journals/event_table_python_v0_oppA.csv"
EVENT_OUT = "/content/drive/MyDrive/Academic-Journals/event_table_topicmatch.csv"

COL_AUTHOR  = "author_id"
COL_JOURNAL = "journal_id"
COL_YEAR    = "t"             # das Jahr t der Zeile
COL_TOPIC   = "topic_match"   # wird von uns befuellt (SPECTER2)

# Spalten, die echte Kommazahlen sind und NICHT zu Int64 werden duerfen
FLOAT_COLS = {COL_TOPIC}

ev = pd.read_csv(EVENT_IN)
print(f"{len(ev):,} Zeilen. Spalten: {list(ev.columns)}")

# --- Ganzzahl-Typen wiederherstellen -------------------------------------
# pandas macht aus jeder Integer-Spalte MIT Leerwerten automatisch float64
# (2020 -> z.B. 2020.0). Beim Zurueckschreiben landet das im CSV. Wir ändern das
# zurück: Jede float-Spalte, deren Werte alle ganzzahlig sind, wird Int64.
restored = []
for c in ev.columns:
    if c in FLOAT_COLS:
        continue
    if pd.api.types.is_float_dtype(ev[c]):
        v = ev[c].dropna()
        if len(v) and np.all(np.mod(v.values, 1) == 0):
            ev[c] = ev[c].astype("Int64")
            restored.append(c)

print("Als Int64 wiederhergestellt:", restored if restored else "(keine)")
print("dtypes:\n", ev.dtypes.to_string())


/tmp/ipykernel_508/1565086609.py:16: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ev = pd.read_csv(EVENT_IN)


6,422,558 Zeilen. Spalten: ['author_id', 'journal_id', 't', 'n_prior_papers', 'coauthor_seed', 't_first_seed', 'first_entry_independent', 'first_entry_ride', 'entering_work_id', 'publisher_id', 'topic_match']
Als Int64 wiederhergestellt: ['t_first_seed']
dtypes:
 author_id                   object
journal_id                  object
t                            int64
n_prior_papers               int64
coauthor_seed                int64
t_first_seed                 Int64
first_entry_independent      int64
first_entry_ride             int64
entering_work_id            object
publisher_id                object
topic_match                float64


### Konsistenz-Check: Passen Event-Tabelle und Embeddings zusammen?

Bevor wir rechnen müssen wir prüfen --> Stammen beide aus derselben Datenbasis? Wir vergleichen nicht
Datei-Hashes (CSV vs. JSONL haben immer verschiedene Hashes), sondern den Inhalt
wie viele unserer eingebetteten Autoren in der Event-Tabelle vorkommen und umgekehrt. Das ist ein Sicherheitscheck --> da wir die Datenbasis nie über das GitHub sharen

In [ ]:
# Überlappung der IDs zwischen Embeddings und Event-Tabelle prüfen --> Ich war mir nicht sicher, ob die Duck-DB-Basis zur Datenbasis von Kevin passt
emb_authors = set(author_works)                      # --> alle produktiven Autoren
emb_journals = {m["journal_id"] for m in meta}
ev_authors = set(ev[COL_AUTHOR].unique())
ev_journals = set(ev[COL_JOURNAL].unique())

# Wie viele unserer produktiven Autoren tauchen in der Event-Tabelle auf?
a_hit = len(emb_authors & ev_authors)
j_hit = len(emb_journals & ev_journals)

print("=== Autoren ===")
print(f"  eingebettet (produktiv)      : {len(emb_authors):,}")
print(f"  davon in der Event-Tabelle   : {a_hit:,} ({a_hit/max(len(emb_authors),1):.1%})")
print("=== Journale ===")
print(f"  in unseren Embeddings        : {len(emb_journals):,}")
print(f"  davon in der Event-Tabelle   : {j_hit:,} ({j_hit/max(len(emb_journals),1):.1%})")

# ID-Format-Check: sehen die IDs ueberhaupt gleich aus?
ex_emb = next(iter(emb_authors))
ex_ev  = next(iter(ev_authors))
print("\n=== ID-Format ===")
print(f"  Embedding-Autor-ID : {ex_emb}")
print(f"  Event-Autor-ID     : {ex_ev}")

if a_hit / max(len(emb_authors), 1) >= 0.9:
    print("\nOK: hohe Überlappung -> gleiche Basis, DuckDB muss NICHT neu gebaut werden.")
elif a_hit == 0:
    print("\nACHTUNG: keine Überlappung. Wahrscheinlich unterschiedliches ID-Format")
    print("  (z.B. volle URL vs. Kurzform) ODER verschiedene Datenversionen.")
    print("  Erst klaeren, bevor gefüllt wird - sonst bleibt alles leer.")
else:
    print("\nTeilweise Überlappung - vor dem Füllen pruefen, woran die Lücke liegt.")

=== Autoren ===
  eingebettet (produktiv)      : 5,500
  davon in der Event-Tabelle   : 0 (0.0%)
=== Journale ===
  in unseren Embeddings        : 64
  davon in der Event-Tabelle   : 0 (0.0%)

=== ID-Format ===
  Embedding-Autor-ID : https://openalex.org/A5052512997
  Event-Autor-ID     : A5090455704

ACHTUNG: keine Überlappung. Wahrscheinlich unterschiedliches ID-Format
  (z.B. volle URL vs. Kurzform) ODER verschiedene Datenversionen.
  Erst klaeren, bevor gefüllt wird - sonst bleibt alles leer.


In [ ]:
def short_id(x):
    return str(x).rsplit("/", 1)[-1]   # "https://openalex.org/A123" -> "A123"

# author_works, meta und row_of auf Kurz-IDs umstellen
author_works = {short_id(a): ws for a, ws in author_works.items()}
for m in meta:
    m["author_id"]  = short_id(m.get("author_id", ""))
    m["journal_id"] = short_id(m["journal_id"])
journal_id_of = {w: short_id(j) for w, j in journal_id_of.items()}

### Diagnosespalten und Statuscodes

Zusaetzlich zu `topic_match` schreibt der Lauf vier Spalten, damit der
Logic- und der Probabilistic-Layer leere Werte unterscheiden koennen:

| Spalte | Bedeutung |
| --- | --- |
| `n_profile_papers` | Paper des Autors **strikt vor t** (im Embedding-Set) |
| `n_journal_papers` | Paper des Journals **strikt vor t** (im Embedding-Set) |
| `profile_cutoff` | tatsaechliches letztes Jahr im Autor-Profil (nicht t-1) |
| `tm_status` | Grund fuer einen leeren Wert |

**Statuscodes**

| Code | Bedeutung |
| --- | --- |
| `ok` | Autor- und Journal-Profil vorhanden, Wert gerechnet |
| `no_author_history` | Autor hat vor t kein eingebettetes Paper |
| `no_journal_history` | **Journal** hat vor t kein eingebettetes Paper |
| `no_author_and_journal_history` | beides fehlt |
| `below_threshold_no_abstract` | Autor hat >= MIN_PAPERS Werke, aber < MIN_PAPERS mit Abstract |
| `below_threshold_unproductive` | Autor hat < MIN_PAPERS Werke insgesamt |
| `author_not_in_db` | Autor-ID nicht in der Rohdatenbasis |

`no_journal_history` ist der bisher unsichtbare Fall. Journal-Profile entstehen
nur aus Papern der ausgewaehlten produktiven Autoren. Hat ein Journal vor t kein
solches Paper, bleibt die Zeile leer - auch wenn der Autor sauber über der
Schwelle liegt und Vorhistorie hat.


In [ ]:
# Jahres-Profile ermitteln, topic_match füllen, sowie die Diagnosespalten
import numpy as np, bisect
from collections import defaultdict

year_of = {m["work_id"]: int(m["year"]) for m in meta if m["year"] is not None}

def build_year_centroids(pairs):
    out = {}
    for k, items in pairs.items():
        items = sorted(items)                       # nach Jahr
        ys = [y for y, _ in items]
        cs = np.cumsum(X[[r for _, r in items]], axis=0)
        out[k] = (ys, cs)
    return out

# Autor-Profile aus den Papern des Autors. Journal-Profile aus allen Embedded-Papern
author_pairs = defaultdict(list)
for aid, ws in author_works.items():
    for w in ws:
        if w in year_of and w in row_of:
            author_pairs[aid].append((year_of[w], row_of[w]))

journal_pairs = defaultdict(list)
for i, m in enumerate(meta):
    if m["year"] is not None and m["journal_id"]:      # None-Journale ausschliessen
        journal_pairs[m["journal_id"]].append((int(m["year"]), i))

A_cent = build_year_centroids(author_pairs)
J_cent = build_year_centroids(journal_pairs)

def centroid_before(entry, year):
    """Zentroid, Anzahl und tatsaechliches Cutoff-Jahr der Paper STRIKT vor `year`."""
    if entry is None:
        return None, 0, None
    ys, cs = entry
    i = bisect.bisect_left(ys, year)                # Paper mit Jahr < year (strikt)
    if i == 0:
        return None, 0, None
    return cs[i - 1] / i, i, ys[i - 1]

# --- Nur Zeilen mit eingebettetem Autor werden überhaupt ausgewertet ----------
embedded = set(author_pairs)
mask = ev[COL_AUTHOR].isin(embedded).values
sub  = ev.loc[mask]
print(f"{mask.sum():,} Zeilen mit eingebettetem Autor (auswertbar), "
      f"{(~mask).sum():,} nicht eingebettet.")

ev_a = sub[COL_AUTHOR].values
ev_j = sub[COL_JOURNAL].values
ev_y = sub[COL_YEAR].astype(int).values

# --- (Key, Jahr) -> Zentroid und Diagnose -----------------------
def unique_centroids(keys_years, cent):
    uniq = sorted(set(keys_years))
    ix   = {k: i for i, k in enumerate(uniq)}
    M    = np.full((len(uniq), X.shape[1]), np.nan)
    N    = np.zeros(len(uniq), dtype=np.int32)
    CUT  = np.full(len(uniq), -1, dtype=np.int32)     # -1 = kein Profil
    for k in uniq:
        c, n, cut = centroid_before(cent.get(k[0]), k[1])
        if c is not None:
            M[ix[k]]   = c
            N[ix[k]]   = n
            CUT[ix[k]] = cut
    return ix, M, N, CUT

ay_ix, AY, AN, ACUT = unique_centroids(zip(ev_a.tolist(), ev_y.tolist()), A_cent)
jy_ix, JY, JN, JCUT = unique_centroids(zip(ev_j.tolist(), ev_y.tolist()), J_cent)
ia = np.array([ay_ix[(a, y)] for a, y in zip(ev_a, ev_y)])
ij = np.array([jy_ix[(j, y)] for j, y in zip(ev_j, ev_y)])

n_prof = AN[ia]          # Autor-Paper strikt vor t
n_jrn  = JN[ij]          # Journal-Paper strikt vor t (nur eingebettete Paper!)
cutoff = ACUT[ia]        # Tatsächliches, letztes Profiljahr des Autors

# --- topic_match rechnen -------------------------------------------------
vals = np.full(len(sub), np.nan)
CH = 200_000
for s in range(0, len(sub), CH):
    e  = slice(s, s + CH)
    d2 = np.sum((AY[ia[e]] - JY[ij[e]]) ** 2, axis=1)
    vals[e] = np.exp(-0.5 * d2 / sig2)

# --- Status pro auswertbarer Zeile ----------------------------------------
has_a = n_prof > 0
has_j = n_jrn  > 0
status_sub = np.where(has_a & has_j, "ok",
             np.where(~has_a & ~has_j, "no_author_and_journal_history",
             np.where(~has_a, "no_author_history", "no_journal_history")))

# --- Status fuer NICHT eingebettete Autoren ------------------------------
# Nicht eingebettet heiss --> Weniger als MIN_PAPERS Paper MIT Titel+Abstract.
# Wir trennen: wirklich unproduktiv vs. an fehlenden Abstracts gescheitert.
cnt_usable = con.execute("""
    SELECT author_id, COUNT(DISTINCT work_id) AS n
    FROM author_paper GROUP BY author_id
""").fetchall()
n_usable = {short_id(a): n for a, n in cnt_usable}

cnt_total = con.execute(f"""
    WITH exploded AS (
        SELECT id AS work_id, unnest(authorships) AS a FROM {TBL}
    )
    SELECT a.author.id AS author_id, COUNT(DISTINCT work_id) AS n
    FROM exploded WHERE a.author.id IS NOT NULL
    GROUP BY a.author.id
""").fetchall()
n_total = {short_id(a): n for a, n in cnt_total}

nonemb_a = ev.loc[~mask, COL_AUTHOR].values
tot = np.array([n_total.get(a, 0)  for a in nonemb_a])
usb = np.array([n_usable.get(a, 0) for a in nonemb_a])
status_non = np.where(tot == 0, "author_not_in_db",
             np.where(tot < MIN_PAPERS, "below_threshold_unproductive",
                                        "below_threshold_no_abstract"))

# --- alles in die Event-Tabelle schreiben --------------------------------
ev[COL_TOPIC]          = np.nan
ev["n_profile_papers"] = 0
ev["n_journal_papers"] = 0
ev["profile_cutoff"]   = pd.NA
ev["tm_status"]        = ""

ev.loc[mask,  COL_TOPIC]          = vals
ev.loc[mask,  "n_profile_papers"] = n_prof
ev.loc[mask,  "n_journal_papers"] = n_jrn
ev.loc[mask,  "profile_cutoff"]   = np.where(cutoff >= 0, cutoff, np.nan)
ev.loc[mask,  "tm_status"]        = status_sub
ev.loc[~mask, "n_profile_papers"] = np.where(usb > 0, 0, 0)   # kein Profil vorhanden
ev.loc[~mask, "tm_status"]        = status_non

for c in ["n_profile_papers", "n_journal_papers", "profile_cutoff"]:
    ev[c] = ev[c].astype("Int64")

# --- Report --------------------------------------------------------------
n = int(ev[COL_TOPIC].notna().sum())
print(f"\ngefuellt: {n:,} von {len(ev):,} ({n/len(ev):.1%})")
if n:
    v = ev[COL_TOPIC].values
    print(f"topic_match: Mittel {np.nanmean(v):.3f}, Median {np.nanmedian(v):.3f}")

print("\n=== tm_status ===")
print(ev["tm_status"].value_counts().to_string())

# Kevins Frage: leere Zeilen, obwohl der Autor ueber der Schwelle liegt
leer_ueber_schwelle = ev.loc[mask & ev[COL_TOPIC].isna(), "tm_status"].value_counts()
print("\n=== leer, obwohl Autor eingebettet (ueber Schwelle) ===")
print(leer_ueber_schwelle.to_string())


1,013,688 Zeilen mit eingebettetem Autor (fuellbar), 5,408,870 nicht eingebettet.

gefuellt: 623,510 von 6,422,558 (9.7%)
topic_match: Mittel 0.776, Median 0.778

=== tm_status ===
tm_status
below_threshold_unproductive     5386995
ok                                623510
no_author_history                 286625
no_author_and_journal_history      67643
no_journal_history                 35910
below_threshold_no_abstract        21875

=== leer, obwohl Autor eingebettet (ueber Schwelle) ===
tm_status
no_author_history                286625
no_author_and_journal_history     67643
no_journal_history                35910


In [ ]:
# Zurueckschreiben - topic_match + Diagnosespalten, alle anderen unverändert
# Int64-Spalten werden von pandas korrekt als "2020" (nicht "2020.0") geschrieben.
ev.to_csv(EVENT_OUT, index=False)
print(f"Geschrieben: {EVENT_OUT}")
print(f"{len(ev):,} Zeilen, {len(ev.columns)} Spalten.")
print("\nNeue Spalten: n_profile_papers, n_journal_papers, profile_cutoff, tm_status")
print("Leeres topic_match ist NIE 0 - der Grund steht in tm_status.")
print("\ndtypes:\n", ev.dtypes.to_string())


Geschrieben: /content/drive/MyDrive/Academic-Journals/event_table_topicmatch.csv
6,422,558 Zeilen, 15 Spalten.

Neue Spalten: n_profile_papers, n_journal_papers, profile_cutoff, tm_status
Leeres topic_match ist NIE 0 - der Grund steht in tm_status.

dtypes:
 author_id                   object
journal_id                  object
t                            int64
n_prior_papers               int64
coauthor_seed                int64
t_first_seed                 Int64
first_entry_independent      int64
first_entry_ride             int64
entering_work_id            object
publisher_id                object
topic_match                float64
n_profile_papers             Int64
n_journal_papers             Int64
profile_cutoff               Int64
tm_status                   object
